In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

In [47]:
BERAU_DATA_PATH = "../datasets/raw/bureau.csv"
BERAU__BALANCE_DATA_PATH = "../datasets/raw/bureau_balance.csv"
IMAGES_PATH = "../images/"
OUTPUT_PATH = "../datasets/preprocess/"
TRAIN_DATA_PATH = "../datasets/raw/application_train.csv"

In [40]:
berau = pd.read_csv(BERAU_DATA_PATH)

In [48]:
app_train = pd.read_csv(TRAIN_DATA_PATH)

In [3]:
berau_balance = pd.read_csv(BERAU__BALANCE_DATA_PATH)

display(berau_balance.head())

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [12]:
berau_balance['STATUS'].unique()

array(['C', '0', 'X', '1', '2', '3', '5', '4'], dtype=object)

In [16]:
# Create a helper column: True if the status indicates any level of overdue (1-5)
berau_balance['IS_OVERDUE'] = berau_balance['STATUS'].isin(['1', '2', '3', '4', '5'])

In [24]:
status_severity = {
    'C': 0,
    '0': 0,
    'X': 0,  
    '1': 1,
    '2': 2,
    '3': 3,
    '4': 4,
    '5': 5,
}

berau_balance['STATUS_SEVERITY'] = berau_balance['STATUS'].map(status_severity)

In [27]:
berau_balance.head(5)

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS,IS_OVERDUE,STATUS_SEVERITY
0,5715448,0,C,False,0
1,5715448,-1,C,False,0
2,5715448,-2,C,False,0
3,5715448,-3,C,False,0
4,5715448,-4,C,False,0


In [37]:
berau_balance_agg = berau_balance.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE':'count',
    'IS_OVERDUE':'sum',
    'STATUS_SEVERITY':'max'
})

berau_balance_agg = berau_balance_agg.rename(columns={
    'MONTHS_BALANCE': 'MONTHS_BALANCE_COUNT',
    'IS_OVERDUE': 'IS_OVERDUE_SUM',
    'STATUS_SEVERITY': 'STATUS_SEVERITY_MAX'
})

# NO necesitas la línea de '_'.join aquí, bórrala

berau_balance_agg = berau_balance_agg.reset_index()

display(berau_balance_agg.head(5))


,SK_ID_BUREAU,MONTHS_BALANCE_COUNT,IS_OVERDUE_SUM,STATUS_SEVERITY_MAX
0,5001709,97,0,0
1,5001710,83,0,0
2,5001711,4,0,0
3,5001712,19,0,0
4,5001713,22,0,0


In [42]:
berau = berau.merge(berau_balance_agg,on='SK_ID_BUREAU',how='left')

In [43]:
berau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,MONTHS_BALANCE_COUNT,IS_OVERDUE_SUM,STATUS_SEVERITY_MAX
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN,NaN,NaN,NaN


In [ ]:
#check teh values NaN 
berau['MONTHS_BALANCE_COUNT'].isna().sum() / len(berau) * 100

54.885727802156566

In [46]:
berau_agg = berau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',                       # cuántos créditos previos tiene el cliente
    'DAYS_CREDIT': ['mean', 'max', 'min'],
    'AMT_CREDIT_SUM': ['mean', 'max', 'sum'],
    'AMT_CREDIT_SUM_DEBT': ['mean', 'sum'],
    'CREDIT_DAY_OVERDUE': ['mean', 'max'],
    'MONTHS_BALANCE_COUNT': ['mean', 'sum'],        # NUEVO: historial mensual promedio/total
    'IS_OVERDUE_SUM': ['mean', 'sum', 'max'],       # NUEVO: meses en atraso, promedio/total/peor caso
    'STATUS_SEVERITY_MAX': 'max',                   # NUEVO: peor nivel de atraso alcanzado en cualquier crédito
})

# Flatten the MultiIndex columns (same technique as before)
berau_agg.columns = ['_'.join(col).upper() for col in berau_agg.columns]
berau_agg = berau_agg.reset_index()

display(berau_agg.head())

,SK_ID_CURR,SK_ID_BUREAU_COUNT,DAYS_CREDIT_MEAN,DAYS_CREDIT_MAX,DAYS_CREDIT_MIN,AMT_CREDIT_SUM_MEAN,AMT_CREDIT_SUM_MAX,AMT_CREDIT_SUM_SUM,AMT_CREDIT_SUM_DEBT_MEAN,AMT_CREDIT_SUM_DEBT_SUM,CREDIT_DAY_OVERDUE_MEAN,CREDIT_DAY_OVERDUE_MAX,MONTHS_BALANCE_COUNT_MEAN,MONTHS_BALANCE_COUNT_SUM,IS_OVERDUE_SUM_MEAN,IS_OVERDUE_SUM_SUM,IS_OVERDUE_SUM_MAX,STATUS_SEVERITY_MAX_MAX
0,100001,7,-735.000000,-49,-1572,207623.571429,378000.0,1453365.000,85240.928571,596686.5,0.0,0,24.571429,172.0,0.142857,1.0,1.0,1.0
1,100002,8,-874.000000,-103,-1437,108131.945625,450000.0,865055.565,49156.200000,245781.0,0.0,0,13.750000,110.0,3.375000,27.0,6.0,1.0
2,100003,4,-1400.750000,-606,-2586,254350.125000,810000.0,1017400.500,0.000000,0.0,0.0,0,NaN,0.0,NaN,0.0,NaN,NaN
3,100004,2,-867.000000,-408,-1326,94518.900000,94537.8,189037.800,0.000000,0.0,0.0,0,NaN,0.0,NaN,0.0,NaN,NaN
4,100005,3,-190.666667,-62,-373,219042.000000,568800.0,657126.000,189469.500000,568408.5,0.0,0,7.000000,21.0,0.000000,0.0,0.0,0.0


In [51]:
#Merge Train with Berau 
app_train = app_train.merge(berau_agg, on='SK_ID_CURR', how='left')


display(app_train.head())

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AMT_CREDIT_SUM_DEBT_MEAN_y,AMT_CREDIT_SUM_DEBT_SUM_y,CREDIT_DAY_OVERDUE_MEAN_y,CREDIT_DAY_OVERDUE_MAX_y,MONTHS_BALANCE_COUNT_MEAN_y,MONTHS_BALANCE_COUNT_SUM_y,IS_OVERDUE_SUM_MEAN_y,IS_OVERDUE_SUM_SUM_y,IS_OVERDUE_SUM_MAX_y,STATUS_SEVERITY_MAX_MAX_y
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,49156.2,245781.0,0.0,0.0,13.75,110.0,3.375,27.0,6.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,NaN,NaN
